# 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용 — 뉴스 키워드 추출기

뉴스 기사에서 핵심 키워드 3개를 **예시와 같은 형식(`키워드: A, B, C`)** 으로 추출한다.

**요구사항 체크**
- [x] `FewShotPromptTemplate` 사용 (문자열 프롬프트 방식, 섹션 4)
- [x] 시스템 메시지 활용 — `FewShotChatMessagePromptTemplate` + `system` 메시지 (구현 힌트 방식, 섹션 5)
- [x] 예시(examples) 최소 3개 → 4개 사용
- [x] 핵심 키워드 3개 추출 + 출력 형식 검증
- [x] 다양한 분야의 뉴스(IT·환경·문화·사회)로 테스트

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)
형식을 일관되게 유지해야 하므로 `temperature`를 낮게 설정한다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm(temperature=0.1)
print(f"{PROVIDER} / {llm.model_name}")

## 3. 예시 데이터 (문제 제공 2개 + 추가 2개) & 테스트 뉴스 (IT·환경·문화·사회)

In [ ]:
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    # ---- 추가 예시 (스포츠 / 경제) ----
    {
        "news": "손흥민이 프리미어리그 경기에서 시즌 10호 골을 기록하며 토트넘의 3대1 승리를 이끌었다. 이로써 손흥민은 9시즌 연속 두 자릿수 득점이라는 기록을 세웠다.",
        "keywords": "손흥민, 프리미어리그, 두자릿수득점",
    },
    {
        "news": "한국은행 금융통화위원회는 물가 상승세가 둔화되고 경기 회복이 더디다는 판단에 따라 기준금리를 0.25%포인트 인하했다. 시장에서는 대출 금리도 함께 낮아질 것으로 전망하고 있다.",
        "keywords": "한국은행, 기준금리, 인하",
    },
]
print(f"예시 개수: {len(examples)}")

In [ ]:
test_news = [
    # 문제에서 주어진 테스트 뉴스 (IT)
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.",
    # 환경
    "기상청은 올여름 평균기온이 관측 이래 가장 높았다고 발표했다. 폭염 일수도 역대 최다를 기록했으며, 전문가들은 기후변화로 인한 이상고온 현상이 앞으로 더 잦아질 것이라고 경고했다.",
    # 문화
    "방탄소년단(BTS)이 완전체로 컴백해 발표한 새 앨범이 빌보드 200 차트 1위에 올랐다. 이번 앨범은 발매 첫 주에만 전 세계에서 200만 장 이상 판매된 것으로 집계됐다.",
    # 사회
    "정부는 저출산 문제 해결을 위해 육아휴직 급여를 월 최대 250만 원으로 인상하는 방안을 발표했다. 맞벌이 부부가 함께 육아휴직을 쓰면 추가 지원금도 지급한다.",
]

## 4. 방법 A — `FewShotPromptTemplate` (문자열 프롬프트)

`prefix`(지시문) + 예시들(`example_prompt`로 하나씩 포맷) + `suffix`(실제 질문) 순서로 하나의 문자열 프롬프트가 만들어진다.

In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

# 예시 1개를 어떤 모양으로 보여줄지 정의
example_prompt = PromptTemplate.from_template("뉴스: {news}\n키워드: {keywords}")

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=("당신은 뉴스 키워드 추출 전문가입니다. "
            "뉴스 기사에서 가장 핵심적인 키워드 3개를 추출하세요. "
            "아래 예시와 똑같이 '키워드: A, B, C' 한 줄 형식으로만 답하세요."),
    suffix="뉴스: {input}\n키워드:",
    input_variables=["input"],
    example_separator="\n\n",
)

# 실제로 LLM에 들어가는 프롬프트 확인
print(few_shot_prompt.format(input="(테스트 뉴스)"))

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_a = few_shot_prompt | llm | StrOutputParser()

# suffix 가 '키워드:' 로 끝나므로 모델이 키워드만 이어 쓰는 경우를 대비해 접두어를 통일
def normalize(text: str) -> str:
    text = text.strip().splitlines()[0].strip()
    return text if text.startswith("키워드:") else f"키워드: {text}"

# 문제에서 주어진 테스트 뉴스(제미나이) 로 확인 → 예상: 키워드: 제미나이, 구글AI, 개발자
print(normalize(chain_a.invoke({"input": test_news[0]})))

## 5. 방법 B — `FewShotChatMessagePromptTemplate` + 시스템 메시지 (구현 힌트)

예시를 `human` → `ai` 대화 쌍으로 넣고, 역할 지시는 `system` 메시지로 분리한다. Chat 모델에는 이 방식이 더 자연스럽다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. 예시 프롬프트 (대화형)
chat_example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])

# 2. Few-Shot 프롬프트
chat_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=chat_example_prompt,
    examples=examples,
)

# 3. 최종 프롬프트 (system + 예시 대화 + 실제 질문)
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요. "
               "반드시 '키워드: A, B, C' 한 줄 형식으로만 답하고 다른 설명은 하지 마세요."),
    chat_few_shot_prompt,
    ("human", "{input}"),
])

# 메시지가 어떻게 구성되는지 확인 (system 1 + 예시 4쌍 8 + human 1 = 10개)
for m in final_prompt.format_messages(input="(테스트 뉴스)"):
    print(f"[{m.type}] {m.content[:40]}")

chain_b = final_prompt | llm | StrOutputParser()

## 6. 다양한 분야의 뉴스로 테스트 (방법 A vs 방법 B 비교)

In [ ]:
def check_format(answer: str) -> bool:
    """'키워드: A, B, C' 형식이고 키워드가 정확히 3개인지 확인"""
    if not answer.startswith("키워드:"):
        return False
    keywords = [k.strip() for k in answer.removeprefix("키워드:").split(",") if k.strip()]
    return len(keywords) == 3


for i, news in enumerate(test_news, 1):
    answer_a = normalize(chain_a.invoke({"input": news}))
    answer_b = normalize(chain_b.invoke({"input": news}))
    print(f"[뉴스 {i}] {news[:45]}...")
    print(f"  방법 A (FewShotPromptTemplate)            → {answer_a}  {'✅' if check_format(answer_a) else '⚠️ 형식 확인'}")
    print(f"  방법 B (FewShotChat + system 메시지)      → {answer_b}  {'✅' if check_format(answer_b) else '⚠️ 형식 확인'}")
    print("-" * 70)